# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors (FAIRˆ²) Exploration with `mlcroissant`

This notebook provides a step-by-step example for loading, exploring, and processing the FAIRˆ² dataset using the [mlcroissant](https://github.com/mlcommons/croissant-python) library.

This dataset contains comprehensive clinical, pathological, and molecular records for 77 cancer survivors who developed a second primary colorectal cancer (CRC), including demographic information, comorbidities, pathological subtypes, MSI-H status, anatomical distribution, and treatment histories.

### Dataset Source
The dataset is provided via a Croissant schema URL:

In [ ]:
# Install `mlcroissant` if necessary
!pip install -q mlcroissant

## 1. Data Loading

Here we use `mlcroissant` to load the dataset metadata and records from the Croissant schema.

**Dataset URL:**
https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json

In [ ]:
import mlcroissant as mlc
import pandas as pd
from pprint import pprint

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the dataset (this downloads and parses the Croissant JSON-LD)
dataset = mlc.Dataset(croissant_url)

# Access metadata: .metadata is an object, so use attribute access
meta = dataset.metadata
print(f"{meta.name}: {meta.description}\n")

## 2. Data Overview

Now we will inspect what record sets and fields are available in this Croissant package.

All entities (record sets, fields, columns) should be referenced using their `@id` field.

In [ ]:
# List all available record sets (@id and name if available)
print("Record Sets available in this dataset:")
record_sets = dataset.record_sets
for rs in record_sets:
    print(f"- @id: {rs['@id']}\n  Name: {rs.get('name', '[no name]')}\n")

# For this dataset, let's get all fields (columns/attributes) from the first record set as an example
if record_sets:
    first_rs_id = record_sets[0]['@id']
    print(f"\nFields in record set with @id = {first_rs_id}:")
    # Retrieve fields (columns) definitions from the record set schema
    fields = dataset.fields(record_set=first_rs_id)
    for field in fields:
        print(f"- Field @id: {field['@id']} | name: {field.get('name', '[no name]')} | Data type: {field.get('dataType', '[unknown]')}")
else:
    print("No record sets detected in this dataset schema.")

## 3. Data Extraction

Now we'll load the records from each record set using their `@id` fields and convert them into pandas DataFrames for analysis.

You can adapt the `record_sets` list to select specific subsets of the data.

In [ ]:
# Extract all data from all record sets, referenced by @id
dataframes = {}

# List of record set @id's
record_set_ids = [rs['@id'] for rs in dataset.record_sets]

# Load each record set into a DataFrame
for rs_id in record_set_ids:
    # Load as a generator of dicts, then to DataFrame
    records = list(dataset.records(record_set=rs_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded {len(df)} records for record set @id: {rs_id}")
    else:
        print(f"No records loaded for record set @id: {rs_id}")

# For demonstration, inspect columns of the main record set (take the first)
if record_set_ids:
    main_rs_id = record_set_ids[0]
    main_df = dataframes[main_rs_id]
    print(f"\nColumns in main record set (@id: {main_rs_id}):")
    print(main_df.columns.tolist())
    display(main_df.head())

## 4. Exploratory Data Analysis (EDA)

Let's perform EDA using available numeric and categorical fields.

We'll filter based on a numeric field and display grouping operations. The demonstration below assumes the presence of a field representing age (replace with an appropriate numeric `@id` if needed).

In [ ]:
# Identify candidate numeric field by @id (update according to the actual field @id in your schema)
# For demonstration, we'll search for a likely 'Age' column:
main_fields = dataset.fields(record_set=main_rs_id)
numeric_field_id = None
for f in main_fields:
    if 'age' in f.get('name','').lower() or 'age' in f['@id'].lower():
        numeric_field_id = f['@id']
        break

if numeric_field_id is None:
    raise ValueError("No field @id for 'age' or suitable numeric field found. Please check schema.")

# View sample data for the chosen field
print(f"Using numeric field @id: {numeric_field_id}")

df = dataframes[main_rs_id]

# Filter records where age > 60 (example threshold)
threshold = 60
filtered_df = df[df[numeric_field_id] > threshold].copy()
print(f"Filtered records with {numeric_field_id} > {threshold}: {len(filtered_df)} records")
display(filtered_df[[numeric_field_id]].head())

# Normalize the age field in the filtered DataFrame
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
display(filtered_df[[numeric_field_id,f"{numeric_field_id}_normalized"]].head())

# Choose group field (e.g., sex, comorbidity, msi_status) by @id
# Search for a categorical field (pick first that matches likely candidates)
group_field = None
for f in main_fields:
    candidate = f.get('name','').lower()
    if any(k in candidate for k in ['sex','gender','msi','status','anatomical']):
        group_field = f['@id']
        break

if group_field and group_field in df.columns:
    grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean()
    print(f"\nGrouped mean {numeric_field_id} by {group_field}:")
    display(grouped_df.head())
else:
    print("No suitable group field found for grouping.")

## 5. Visualization

Now, let's visualize the distribution and relationships for key fields.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field (e.g., Age)
plt.figure(figsize=(8,5))
sns.histplot(df[numeric_field_id], bins=12, kde=True)
plt.xlabel(numeric_field_id)
plt.title(f"Distribution of {numeric_field_id}")
plt.tight_layout()
plt.show()

# Boxplot of numeric field by group (if group_field exists)
if group_field and group_field in df.columns:
    plt.figure(figsize=(8,5))
    sns.boxplot(x=df[group_field], y=df[numeric_field_id])
    plt.xlabel(group_field)
    plt.ylabel(numeric_field_id)
    plt.title(f"{numeric_field_id} by {group_field}")
    plt.tight_layout()
    plt.show()
else:
    print("No group field found for boxplot.")

## 6. Conclusion

In this notebook, we've demonstrated how to load, explore, and perform initial analysis of the FAIRˆ² colorectal cancer survivors dataset using the mlcroissant library. By referencing all dataset entities by their `@id`, we ensured precise and schema-compliant data access. Further analysis could involve supervised modeling, feature selection, or clinical outcome association studies based on the rich phenotypic and molecular data in the record sets.

For more information on the dataset or to view/extend the Croissant schema programmatically, see:  
https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json